<a href="https://colab.research.google.com/github/charansai12322/skillmapagent/blob/main/skillmapagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain

In [2]:
!pip install -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [3]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('sec1')
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=google_api_key
)

In [4]:
!pip install langchain-tavily

In [5]:
from langchain_tavily import TavilySearch
from google.colab import userdata

tavily_api_key = userdata.get('TAVILY_API_KEY')

skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)

In [6]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPIDAPI_KEY')

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

In [7]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [8]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [9]:
config = {"configurable": {"thread_id": "1"}}

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[skill_demand_tool, search_jobs],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
    debug=True  # ← Add this!

)

In [16]:
user_query = "What's the demand for generative AI in the industry and show me related job openings in India"

response = agent.invoke({
  "messages": [{"role": "user", "content": user_query}]
}, config=config)

print(response["messages"][-1].content)

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='02d85860-0977-40c4-b7f6-e0b2419991d0')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "generative AI"}'}, '__gemini_function_call_thought_signatures__': {'56ac590a-7bc4-4d71-a934-c3a8c1b492b6': 'CqIGAWkUfROe6KU/mxWTAcC1I36u/ZXkHgYjYIH5RD6CUF6RKQrCV50M6FWYT7AZKWJwQAck2Lwn1e+tqhQ69eXJzetCJQV4AsnbgDiWu8NfmrxGkqX5fv2ZpThHyZ14U0eBdtKYDMzr8JqL5R6fm0bio5ysx+2eRok5uL04BWSq5yhMfvoY6cnMeH3rSt4JrMgsjS2hfSliLD8qEitvtOi6KUpuI4FibuiSnaqXNEab4YdQAZHIaYiKjzilE3RWGqgvIZNnS9TDbjzP3VPE7ol34ftlS9i/nSzueW8GOoQrQ1lBQ0oJIsoVjBMjYx6Ewe1M+zX5UAniA7RlaZCRkhjesyyyiKbxyLd8jG5sNa+F0W0n3Kvdlwy38RCux2k1JN1gcGdqJJTaXLRuvAXtHIaDgPMCdv7jdeSwHh0xypzLco/xpif9VKbDOzi1G/g5wn5RIv0phi4vPJJid+3e50qGZchXejcMkuluiyskhiXw/M56uFqT